# Bab 10. pandas: Data Berlabel dan Bertipe Campuran

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import pandas as pd
import numpy as np
from siapkan import kue_tiga_bulan

produk, trans, g = kue_tiga_bulan()

## 1. Membuat DataFrame produk

In [ ]:
import pandas as pd
import numpy as np

produk = pd.DataFrame({
    "id_produk": ["P1", "P2", "P3", "P4", "P5", "P6"],
    "nama": ["Nastar", "Kastengel", "Putri Salju",
             "Brownies", "Bolu Pandan", "Lapis Legit"],
    "kategori": ["kering", "kering", "kering",
                 "basah", "basah", "basah"],
    "harga": [85000, 95000, 78000, 45000, 40000, 120000],
})
print(produk)

Keluaran yang diharapkan:

```
  id_produk         nama kategori   harga
0        P1       Nastar   kering   85000
1        P2    Kastengel   kering   95000
2        P3  Putri Salju   kering   78000
3        P4     Brownies    basah   45000
4        P5  Bolu Pandan    basah   40000
5        P6  Lapis Legit    basah  120000
```

## 2. Memeriksa tipe

In [ ]:
print(produk.dtypes)

Keluaran yang diharapkan:

```
id_produk      str
nama           str
kategori       str
harga        int64
dtype: object
```

## 3. Memeriksa data yang hilang

In [ ]:
print(trans.head())
print(trans.isna().sum())
print(trans["jumlah"].dtype)

Keluaran yang diharapkan:

```
     tanggal id_produk  jumlah
0 2025-11-03        P1     5.0
1 2025-11-03        P4     4.0
2 2025-11-08        P2    20.0
3 2025-11-08        P5    13.0
4 2025-11-15        P1     NaN

tanggal      0
id_produk    0
jumlah       2
float64
```

## 4. Tipe bilangan bulat nullable

In [ ]:
print(trans["jumlah"].astype("Int64").head(6))

Keluaran yang diharapkan:

```
0       5
1       4
2      20
3      13
4    <NA>
5      15
```

## 5. Membuang baris tak lengkap

In [ ]:
t = trans.dropna(subset=["jumlah"]).copy()
t["jumlah"] = t["jumlah"].astype("int64")
print(t.shape)

Keluaran yang diharapkan:

```
(28, 3)
```

## 6. Tiga cara seleksi

In [ ]:
t["jumlah"]              # satu kolom, jadi Series
t[["nama", "jumlah"]]    # beberapa kolom, jadi DataFrame

t.loc[3, "jumlah"]       # baris berlabel 3, kolom "jumlah"
t.loc[t["jumlah"] > 20]  # penyaringan boolean

t.iloc[0]                # baris pertama menurut posisi
t.iloc[0:3, 0:2]         # irisan menurut posisi

## 7. Pola yang tidak bekerja

In [ ]:
t[t["jumlah"] > 20]["jumlah"] = 0    # TIDAK berpengaruh

## 8. Cara yang benar

In [ ]:
t.loc[t["jumlah"] > 20, "jumlah"] = 0

## 9. Menggabungkan dua tabel

In [ ]:
g = t.merge(produk, on="id_produk", how="inner")
g["omzet"] = g["jumlah"] * g["harga"]

print(g[["tanggal", "nama", "kategori",
         "jumlah", "omzet"]].head(4))

Keluaran yang diharapkan:

```
     tanggal         nama kategori  jumlah    omzet
0 2025-11-03       Nastar   kering       5   425000
1 2025-11-03     Brownies    basah       4   180000
2 2025-11-08    Kastengel   kering      20  1900000
3 2025-11-08  Bolu Pandan    basah      13   520000
```

## 10. Membandingkan jenis join

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    j = t.merge(produk, on="id_produk", how=how)
    print(how, len(j),
          j["nama"].isna().sum(),
          j["jumlah"].isna().sum())

## 11. Meringkas per kategori

In [ ]:
ring = (g.groupby("kategori")["omzet"]
         .agg(["count", "sum", "mean"]))
print(ring.round(0).astype("int64"))

Keluaran yang diharapkan:

```
          count       sum     mean
kategori
basah         8   3240000   405000
kering       19  24027000  1264579
```

## 12. Peringkat produk

In [ ]:
per = (g.groupby("nama")["omzet"].sum()
        .sort_values(ascending=False))
print(per)

Keluaran yang diharapkan:

```
nama
Kastengel      10165000
Nastar          7310000
Putri Salju     6552000
Bolu Pandan     1800000
Brownies        1440000
Name: omzet, dtype: int64
```

## 13. Omzet per bulan per kategori

In [ ]:
bln = (g.set_index("tanggal")
        .groupby([pd.Grouper(freq="ME"), "kategori"])["omzet"]
        .sum()
        .unstack())
print(bln)

Keluaran yang diharapkan:

```
kategori      basah   kering
tanggal
2025-11-30   790000  6644000
2025-12-31   750000  9645000
2026-01-31  1700000  7738000
```

## 14. Grafik cepat dari DataFrame

In [ ]:
ax = (bln / 1e6).plot.bar(figsize=(5.3, 2.6))
ax.set_ylabel("omzet (juta rupiah)")